In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *

In [2]:
from connect import bob

sftp = bob()

folder = '/data/slam/home/ulyanov/sdo/'
files = sorted(sftp.listdir(folder))

In [3]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 512

view_north = View(nx, ny, xc, yc, Rsun, -90, 90, 0) ### North pole view
grid = np.mgrid[:nx,:ny].astype(np.float32)

mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])


local_file = 'temp.fits'
for file in files[:]:
    print(file)
    remote_file = folder + file

    sftp.get(remote_file, local_file)

    with fits.open(local_file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    #data = rebin(data, 4, update_header=header)
    view = View.from_header(header)

    transform = view_north.to_spherical(correct_mu=True, mu_thr=0.1) - view.to_spherical(correct_mu=True, correct_dr=True, mu_thr=0.1)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    n = (~np.isnan(data)) * (1 / alpha) ** 2
    coverage += np.nan_to_num(n)
    mean_ = mean.copy()
    mean += np.nan_to_num((data - mean) * n / coverage)
    variance += np.nan_to_num((data - mean) * (data - mean_) * n)

variance = variance / coverage
mean[coverage == 0] = np.nan
coverage[coverage == 0] = np.nan
variance[coverage == 0] = np.nan

hmi.M_720s.20250501_000000_TAI.3.magnetogram.fits
hmi.M_720s.20250501_001200_TAI.3.magnetogram.fits
hmi.M_720s.20250501_002400_TAI.3.magnetogram.fits
hmi.M_720s.20250501_003600_TAI.3.magnetogram.fits
hmi.M_720s.20250501_004800_TAI.3.magnetogram.fits
hmi.M_720s.20250501_010000_TAI.3.magnetogram.fits
hmi.M_720s.20250501_011200_TAI.3.magnetogram.fits
hmi.M_720s.20250501_012400_TAI.3.magnetogram.fits
hmi.M_720s.20250501_013600_TAI.3.magnetogram.fits
hmi.M_720s.20250501_014800_TAI.3.magnetogram.fits
hmi.M_720s.20250501_020000_TAI.3.magnetogram.fits
hmi.M_720s.20250501_021200_TAI.3.magnetogram.fits
hmi.M_720s.20250501_022400_TAI.3.magnetogram.fits
hmi.M_720s.20250501_023600_TAI.3.magnetogram.fits
hmi.M_720s.20250501_024800_TAI.3.magnetogram.fits
hmi.M_720s.20250501_030000_TAI.3.magnetogram.fits
hmi.M_720s.20250501_031200_TAI.3.magnetogram.fits
hmi.M_720s.20250501_032400_TAI.3.magnetogram.fits
hmi.M_720s.20250501_033600_TAI.3.magnetogram.fits
hmi.M_720s.20250501_034800_TAI.3.magnetogram.fits


/tmp/ipykernel_109150/4219738304.py:35: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [4]:
show_polar_data(mean, vmin=-20, vmax=20, cmap='seismic', label=r'$B_{los}, G$')

(<Figure size 1000x1000 with 2 Axes>, <Axes: >)

In [5]:
show_polar_data(np.sqrt(variance), vmin=0, vmax=20, cmap='turbo', label=r'$\sigma_B, G$')

/tmp/ipykernel_109150/3381950695.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_polar_data(np.sqrt(variance), vmin=0, vmax=20, cmap='turbo', label=r'$\sigma_B, G$')


(<Figure size 1000x1000 with 2 Axes>, <Axes: >)